# Salary Analysis EDA

Notebook này đọc và khám phá file `data/analysis/salary_analysis_clean.csv`.

Mục tiêu: hiểu dataset salary numeric đã lọc sạch, đồng thời học các pandas method hay dùng cho phân tích dữ liệu lương.

In [ ]:
from pathlib import Path
import ast
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

try:
    from IPython.display import display
except Exception:  # pragma: no cover - notebook convenience fallback
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string(index=False))
        else:
            print(obj)


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "analysis" / "salary_analysis_clean.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing data/analysis/salary_analysis_clean.csv")


REPO_ROOT = find_repo_root()
SALARY_PATH = REPO_ROOT / "data" / "analysis" / "salary_analysis_clean.csv"

print(f"Repo root: {REPO_ROOT}")
print(f"Salary analysis file: {SALARY_PATH}")

## 1. Load dữ liệu

Pandas cần học: `pd.read_csv`, `.shape`, `.columns`, `.head()`.

`encoding="utf-8-sig"` giúp đọc CSV có BOM tốt hơn, đặc biệt khi file từng được mở bằng Excel.

In [ ]:
if not SALARY_PATH.exists():
    raise FileNotFoundError(SALARY_PATH)

salary = pd.read_csv(SALARY_PATH, encoding="utf-8-sig")

print(f"Rows: {len(salary):,}")
print(f"Columns: {len(salary.columns):,}")
display(pd.DataFrame({"column": salary.columns}))
display(salary.head(10))

## 2. Chuẩn hóa kiểu dữ liệu cho EDA

File đầu vào đã là salary-clean dataset. Cell này không ghi đè file, chỉ tạo `salary_eda` để phân tích.

Pandas cần học: `.copy()`, `.astype("string")`, `.str.strip()`, `pd.to_numeric`, `pd.to_datetime(..., format="mixed")`.

In [ ]:
salary_eda = salary.copy()

text_columns = [
    "source",
    "url",
    "job_id",
    "title",
    "company",
    "location",
    "salary_raw",
    "salary_currency",
    "skills",
    "experience_raw",
    "description",
    "parse_status",
    "posted_raw",
    "location_cities",
    "seniority",
    "work_mode",
    "employment_type",
    "salary_period",
    "_input_file",
    "salary_suspicious_reason",
    "salary_label",
]
numeric_columns = ["salary_min", "salary_max", "experience_min", "experience_max", "salary_midpoint", "_salary_midpoint"]
date_columns = ["scraped_at", "posted_at", "valid_through", "scraped_at_parsed", "posted_at_parsed", "valid_through_parsed"]


def clean_text_series(series):
    cleaned = series.astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    return cleaned.mask(cleaned.eq(""), pd.NA)


for column in text_columns:
    if column in salary_eda.columns:
        salary_eda[column] = clean_text_series(salary_eda[column])

for column in numeric_columns:
    if column in salary_eda.columns:
        salary_eda[column] = pd.to_numeric(salary_eda[column], errors="coerce")

for column in date_columns:
    if column in salary_eda.columns:
        salary_eda[f"{column}_dt"] = pd.to_datetime(salary_eda[column], errors="coerce", utc=True, format="mixed")

display(salary_eda.dtypes.astype(str).rename_axis("column").reset_index(name="dtype"))

## 3. Quality checks

Ta xác nhận lại rằng đây đúng là dataset salary numeric đã lọc sạch.

Pandas cần học: `.isna()`, `.notna()`, `.duplicated()`, `.eq()`, boolean masks, và `assert` để notebook có kiểm chứng.

In [ ]:
has_numeric_bound = salary_eda[["salary_min", "salary_max"]].notna().any(axis=1)
suspicious_reason = salary_eda["salary_suspicious_reason"].fillna("").astype("string").str.strip()

quality_checks = pd.DataFrame(
    [
        {"check": "rows", "value": len(salary_eda)},
        {"check": "unique_urls", "value": salary_eda["url"].nunique()},
        {"check": "duplicate_urls", "value": int(salary_eda["url"].duplicated().sum())},
        {"check": "salary_label_values", "value": ", ".join(sorted(salary_eda["salary_label"].dropna().unique()))},
        {"check": "rows_with_numeric_bound", "value": int(has_numeric_bound.sum())},
        {"check": "missing_currency", "value": int(salary_eda["salary_currency"].isna().sum())},
        {"check": "missing_period", "value": int(salary_eda["salary_period"].isna().sum())},
        {"check": "non_empty_suspicious_reason", "value": int(suspicious_reason.ne("").sum())},
        {"check": "salary_min_non_positive", "value": int(salary_eda["salary_min"].notna().mul(salary_eda["salary_min"].le(0)).sum())},
        {
            "check": "salary_min_gt_salary_max",
            "value": int(
                (
                    salary_eda["salary_min"].notna()
                    & salary_eda["salary_max"].notna()
                    & salary_eda["salary_min"].gt(salary_eda["salary_max"])
                ).sum()
            ),
        },
    ]
)

assert len(salary_eda) > 0, "salary_analysis_clean.csv must contain rows"
assert salary_eda["url"].is_unique, "Salary dataset should contain one row per URL"
assert salary_eda["salary_label"].eq("numeric").all(), "Every row should have salary_label == numeric"
assert suspicious_reason.eq("").all(), "Salary dataset should exclude suspicious salary rows"
assert has_numeric_bound.all(), "Every row should have salary_min or salary_max"
assert salary_eda.loc[salary_eda["salary_min"].notna(), "salary_min"].gt(0).all(), "salary_min should be positive when present"
assert not (salary_eda["salary_min"].notna() & salary_eda["salary_max"].notna() & salary_eda["salary_min"].gt(salary_eda["salary_max"])).any()
assert salary_eda["salary_currency"].notna().all(), "Numeric salary rows should have salary_currency"
assert salary_eda["salary_period"].notna().all(), "Numeric salary rows should have salary_period"

display(quality_checks)

## 4. Missing values và coverage

Không phải cột nào cũng cần đầy đủ. Ví dụ `experience_max` có thể thiếu vì nhiều job chỉ ghi số năm tối thiểu.

Pandas cần học: `.isna().sum()`, `.notna().sum()`, `.mean()` trên boolean để tính tỷ lệ.

In [ ]:
coverage = pd.DataFrame(
    {
        "column": salary_eda.columns,
        "non_null": [int(salary_eda[column].notna().sum()) for column in salary_eda.columns],
        "missing": [int(salary_eda[column].isna().sum()) for column in salary_eda.columns],
        "missing_rate": [round(float(salary_eda[column].isna().mean()), 4) for column in salary_eda.columns],
        "dtype": [str(salary_eda[column].dtype) for column in salary_eda.columns],
    }
).sort_values(["missing_rate", "column"], ascending=[False, True])

display(coverage.head(30))

## 5. Tổng quan theo source, currency, period

Salary không nên trộn USD với VND, hoặc month với year. Vì vậy các summary bên dưới luôn group theo `salary_currency` và `salary_period`.

Pandas cần học: `.groupby()`, `.agg()`, `.reset_index()`, `.sort_values()`.

In [ ]:
source_summary = (
    salary_eda.groupby("source", dropna=False)
    .agg(
        jobs=("url", "count"),
        companies=("company", "nunique"),
        median_midpoint=("salary_midpoint", "median"),
        min_midpoint=("salary_midpoint", "min"),
        max_midpoint=("salary_midpoint", "max"),
        median_experience_min=("experience_min", "median"),
    )
    .reset_index()
    .sort_values("jobs", ascending=False)
)

currency_period_summary = (
    salary_eda.groupby(["source", "salary_currency", "salary_period"], dropna=False)
    .agg(
        jobs=("url", "count"),
        companies=("company", "nunique"),
        median_midpoint=("salary_midpoint", "median"),
        p25_midpoint=("salary_midpoint", lambda s: s.quantile(0.25)),
        p75_midpoint=("salary_midpoint", lambda s: s.quantile(0.75)),
        min_midpoint=("salary_midpoint", "min"),
        max_midpoint=("salary_midpoint", "max"),
    )
    .reset_index()
    .sort_values(["source", "salary_currency", "salary_period"])
)

display(source_summary)
display(currency_period_summary)

## 6. Salary distribution

Cell này xem phân phối salary midpoint theo từng nhóm currency/period. Không dùng một median chung cho tất cả vì đơn vị khác nhau.

Pandas cần học: `.describe()`, `.pivot_table()`, và quantile.

In [ ]:
salary_distribution = (
    salary_eda.groupby(["salary_currency", "salary_period"], dropna=False)["salary_midpoint"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    .reset_index()
)

median_pivot = salary_eda.pivot_table(
    index="source",
    columns=["salary_currency", "salary_period"],
    values="salary_midpoint",
    aggfunc="median",
)

count_pivot = salary_eda.pivot_table(
    index="source",
    columns=["salary_currency", "salary_period"],
    values="url",
    aggfunc="count",
    fill_value=0,
)

display(salary_distribution)
display(median_pivot)
display(count_pivot)

## 7. Optional charts

Project không khai báo `matplotlib` trong `requirements.txt`, nên cell này chỉ vẽ nếu môi trường đã có thư viện.

Pandas cần học: `.unstack()` để chuyển groupby result thành bảng phù hợp cho chart.

In [ ]:
try:
    get_ipython()
    in_ipython = True
except NameError:
    in_ipython = False

if not in_ipython:
    plt = None
    print("Running outside IPython/Jupyter; skip charts during validation.")
else:
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        plt = None
        print(f"matplotlib is unavailable; skip charts. Reason: {exc}")

if in_ipython and plt is not None:
    try:
        jobs_by_source_currency = salary_eda.groupby(["source", "salary_currency"]).size().unstack(fill_value=0)
        jobs_by_source_currency.plot(kind="bar", title="Salary samples by source and currency")
        plt.ylabel("jobs")
        plt.tight_layout()
        plt.show()

        for (currency, period), group in salary_eda.groupby(["salary_currency", "salary_period"]):
            group.boxplot(column="salary_midpoint", by="source")
            plt.title(f"Salary midpoint by source - {currency}/{period}")
            plt.suptitle("")
            plt.ylabel(f"{currency} per {period}")
            plt.tight_layout()
            plt.show()
    except Exception as exc:
        print(f"Could not render charts in this environment; skip charts. Reason: {exc}")

## 8. Seniority, work mode, employment type

Các nhóm này giúp hiểu salary samples đang nghiêng về level hoặc kiểu làm việc nào.

Pandas cần học: groupby nhiều cột và filter nhóm có sample size đủ lớn trước khi diễn giải.

In [ ]:
def grouped_salary_summary(group_columns, min_jobs=3):
    summary = (
        salary_eda.groupby([*group_columns, "salary_currency", "salary_period"], dropna=False)
        .agg(
            jobs=("url", "count"),
            companies=("company", "nunique"),
            median_midpoint=("salary_midpoint", "median"),
            p25_midpoint=("salary_midpoint", lambda s: s.quantile(0.25)),
            p75_midpoint=("salary_midpoint", lambda s: s.quantile(0.75)),
        )
        .reset_index()
    )
    return summary.loc[summary["jobs"].ge(min_jobs)].sort_values(["salary_currency", "salary_period", "jobs"], ascending=[True, True, False])


display(grouped_salary_summary(["seniority"], min_jobs=3))
display(grouped_salary_summary(["work_mode"], min_jobs=3))
display(grouped_salary_summary(["employment_type"], min_jobs=3))

## 9. Experience buckets

Ta bucket hóa `experience_min` để nhìn phân bố dễ hơn. Đây là phân tích, không phải thay đổi parser.

Pandas cần học: `pd.cut()` để biến số liên tục thành nhóm.

In [ ]:
salary_eda["experience_bucket"] = pd.cut(
    salary_eda["experience_min"],
    bins=[-1, 0, 1, 3, 5, 10, 100],
    labels=["0", "0-1", "1-3", "3-5", "5-10", "10+"],
)
salary_eda["experience_bucket"] = salary_eda["experience_bucket"].astype("string").fillna("unknown")

experience_summary = grouped_salary_summary(["experience_bucket"], min_jobs=3)
display(salary_eda["experience_bucket"].value_counts(dropna=False).rename_axis("experience_bucket").reset_index(name="jobs"))
display(experience_summary)

## 10. Skills salary samples

Cột `skills` đọc từ CSV có thể là chuỗi list. Ta parse thành list rồi `.explode()` để một dòng là một job-skill.

Pandas cần học: `.apply()`, `.explode()`, `.drop_duplicates()`, `.map()`.

In [ ]:
def fold_text(value):
    if pd.isna(value):
        return ""
    text = str(value)
    text = "".join(
        character
        for character in unicodedata.normalize("NFD", text)
        if unicodedata.category(character) != "Mn"
    )
    return re.sub(r"\s+", " ", text).strip().casefold()


def to_listish(value):
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed if str(item).strip()]
        except (ValueError, SyntaxError):
            pass
    return [part.strip() for part in text.replace(";", ",").split(",") if part.strip()]


skill_alias = {
    "js": "javascript",
    "reactjs": "react",
    "react.js": "react",
    "vuejs": "vue",
    "vue.js": "vue",
    "node": "node.js",
    "nodejs": "node.js",
    "golang": "go",
    "py": "python",
    "postgres": "postgresql",
    "k8s": "kubernetes",
    "ts": "typescript",
    "csharp": "c#",
    "dotnet": ".net",
    "ci cd": "ci/cd",
    "html5": "html",
    "css3": "css",
    "nextjs": "next.js",
    "nestjs": "nest.js",
}


def normalize_skill(value):
    folded = fold_text(value).replace("&", " and ")
    folded = re.sub(r"\s+", " ", folded).strip()
    return skill_alias.get(folded, folded)


salary_skills = (
    salary_eda[["url", "source", "title", "company", "salary_currency", "salary_period", "salary_midpoint", "skills"]]
    .assign(skill=salary_eda["skills"].apply(to_listish))
    .explode("skill", ignore_index=True)
)
salary_skills["skill"] = clean_text_series(salary_skills["skill"])
salary_skills["skill_norm"] = salary_skills["skill"].map(normalize_skill).astype("string")
salary_skills = (
    salary_skills.loc[salary_skills["skill_norm"].notna() & salary_skills["skill_norm"].ne("")]
    .drop_duplicates(["url", "skill_norm"])
    .reset_index(drop=True)
)

skill_salary_summary = (
    salary_skills.groupby(["skill_norm", "salary_currency", "salary_period"], dropna=False)
    .agg(
        salary_samples=("url", "nunique"),
        companies=("company", "nunique"),
        median_midpoint=("salary_midpoint", "median"),
        p25_midpoint=("salary_midpoint", lambda s: s.quantile(0.25)),
        p75_midpoint=("salary_midpoint", lambda s: s.quantile(0.75)),
    )
    .reset_index()
)

display(salary_skills.head(20))
display(skill_salary_summary.loc[skill_salary_summary["salary_samples"].ge(5)].sort_values(["salary_samples", "skill_norm"], ascending=[False, True]).head(40))

## 11. Top salary và bottom salary để review

Dù dataset đã lọc suspicious rows, extreme values vẫn nên được review trước khi kết luận.

Pandas cần học: `.sort_values()`, `.groupby()`, lấy top/bottom theo từng nhóm.

In [ ]:
review_columns = [
    "source",
    "title",
    "company",
    "salary_raw",
    "salary_min",
    "salary_max",
    "salary_midpoint",
    "salary_currency",
    "salary_period",
    "experience_raw",
    "url",
]

top_salary_by_group = (
    salary_eda.sort_values("salary_midpoint", ascending=False)
    .groupby(["salary_currency", "salary_period"], dropna=False)
    .head(10)
    .sort_values(["salary_currency", "salary_period", "salary_midpoint"], ascending=[True, True, False])
)

bottom_salary_by_group = (
    salary_eda.sort_values("salary_midpoint", ascending=True)
    .groupby(["salary_currency", "salary_period"], dropna=False)
    .head(10)
    .sort_values(["salary_currency", "salary_period", "salary_midpoint"], ascending=[True, True, True])
)

display(top_salary_by_group[review_columns])
display(bottom_salary_by_group[review_columns])

## 12. Câu hỏi phân tích tiếp theo

Các câu hỏi nên trả lời sau khi đã hiểu file này:

- Median salary theo source có khác nhau không, hay do sample mix khác nhau?
- USD/month và VND/month có nên quy đổi chung không? Nếu có, cần chọn tỷ giá và ghi rõ ngày.
- Những skill có salary cao có đủ sample size chưa?
- Salary theo seniority có hợp lý không, hay parser đang suy luận seniority quá rộng?
- Các row top/bottom salary có phải job thật hay parser cần cải thiện?